# Teacher feedback generation with Qwen3.5-27B-FP8 (single L40S)

This notebook uses a large **teacher** model to write tutor-style feedback for every row of `training_data_moh_eng_rubrics.csv`. The output becomes the QLoRA fine-tuning set for a smaller **student** model.

**Why `Qwen/Qwen3.5-27B-FP8`**
- **Fits the hardware.** It is the strongest open model (Apache-2.0) that runs on one 48 GB L40S. The official FP8 weights take about 29 GB and lose almost no quality. The L40S (Ada) supports FP8 natively, which leaves memory for large batches. Qwen3.5's hybrid linear attention keeps the KV cache small. Larger options such as Qwen3.5-122B-A10B do not fit.
- **Writing quality.** On Qwen's benchmarks the 27B dense model beats the 35B-A3B MoE at instruction following and writing. Faithful, specific feedback matters more here than raw speed. The MoE remains a faster fallback.
- **Same family as the student.** The student is likely Qwen3.5-4B or 9B. A same-family teacher shares the tokenizer, chat template and style, which makes distillation cleaner.

**Pipeline** (resumable; all state is checkpointed to JSONL)
1. Load and validate the rubric dataset.
2. Generate feedback with vLLM, batched and conditioned on the per-criterion MET/NOT MET marks.
3. Quality-gate every row:
   - Rule checks: length, truncation, markdown, mentions of grading mechanics.
   - An LLM judge pass that checks faithfulness to the marks and to the student's answer.
4. Regenerate failed rows with a new seed, up to `MAX_ATTEMPTS` times.
5. Export a CSV with a `feedback` column, plus chat-format `sft_train.jsonl` / `sft_val.jsonl` for QLoRA. The train/val split is by question.

> Run this on the cluster (Linux + CUDA), since vLLM does not support Windows. Use a fresh virtual environment, because vLLM pins its own PyTorch build.

In [1]:
# Run once in a fresh environment, then restart the kernel.
# Qwen3.5 needs a recent vLLM; pinned for reproducibility.
import sys
!{sys.executable} -m pip install -q "vllm==0.31.0" pandas


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [2]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import random
import re
import time
from pathlib import Path

import pandas as pd

# Weights are ~30 GB. On clusters with a small home quota, point the Hugging Face cache at scratch storage.
# os.environ["HF_HOME"] = "/scratch/<user>/hf_cache"

BASE_DIR = Path.cwd()
if BASE_DIR.name != "feedback" and (BASE_DIR / "feedback").is_dir():
    BASE_DIR = BASE_DIR / "feedback"

DATASET_NAME = "training_data_moh_eng_rubrics"
INPUT_PATH = BASE_DIR / "Datasets" / f"{DATASET_NAME}.csv"
OUTPUT_DIR = BASE_DIR / "Datasets" / "teacher_feedback"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GENERATIONS_PATH = OUTPUT_DIR / f"{DATASET_NAME}_generations.jsonl"
JUDGEMENTS_PATH = OUTPUT_DIR / f"{DATASET_NAME}_judgements.jsonl"
OUTPUT_CSV_PATH = OUTPUT_DIR / f"{DATASET_NAME}_with_feedback.csv"
SFT_TRAIN_PATH = OUTPUT_DIR / "sft_train.jsonl"
SFT_VAL_PATH = OUTPUT_DIR / "sft_val.jsonl"

TEACHER_MODEL = "Qwen/Qwen3.5-27B-FP8"  # faster, slightly weaker fallback: "Qwen/Qwen3.5-35B-A3B-FP8"
TEACHER_THINKING = False  # True reasons before writing: better on hard rows, ~5-10x more tokens
JUDGE_THINKING = True  # without reasoning the judge misreads student answers and rejects good feedback
MAX_MODEL_LEN = 16384 if (TEACHER_THINKING or JUDGE_THINKING) else 4096
GPU_MEMORY_UTILIZATION = 0.90
MAX_NUM_SEQS = 128  # concurrent sequences; Qwen3.5 needs one Mamba cache block each (~168 fit on an L40S at 0.90)
MAX_FEEDBACK_TOKENS = 8192 if TEACHER_THINKING else 512
MAX_JUDGE_TOKENS = 8192 if JUDGE_THINKING else 512
MAX_ATTEMPTS = 4  # generations per row before giving up on passing the checks; each retry also changes the style
CHUNK_SIZE = 512  # rows per vLLM call; results are checkpointed after each chunk
SEED = 42

VAL_FRACTION = 0.1  # fraction of *questions* held out for the SFT validation split
SFT_INCLUDE_CRITERION_RESULTS = True  # whether the student model sees the MET/NOT MET marks in its prompt

print(f"Input:   {INPUT_PATH}")
print(f"Output:  {OUTPUT_DIR}")
print(f"Teacher: {TEACHER_MODEL} (thinking={TEACHER_THINKING}, judge thinking={JUDGE_THINKING})")

Input:   /common/home/projectgrps/CS480/CS480G3/Datasets/training_data_moh_eng_rubrics.csv
Output:  /common/home/projectgrps/CS480/CS480G3/Datasets/teacher_feedback
Teacher: Qwen/Qwen3.5-27B-FP8 (thinking=False, judge thinking=True)


## Load and validate the dataset

`id` is read as a string because `1.1` and `1.10` are different questions. Ids also collide across the merged source sets (16 ids map to more than one question), so later steps key on row position and question text, never on `id`.

In [3]:
EXPECTED_COLUMNS = [
    "id",
    "question",
    "desired_answer",
    "student_answer",
    "criteria[]",
    "criteria_score[]",
    "criteria_max_score[]",
    "student_score",
]


def load_and_validate_dataset(path: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = [column for column in EXPECTED_COLUMNS if column not in raw.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    parsed = raw.copy()
    # Some source rows use Windows line endings; normalise so training and inference prompts match.
    for column in ["question", "desired_answer", "student_answer"]:
        parsed[column] = parsed[column].str.replace("\r\n", "\n").str.replace("\r", "\n")
    parsed["criteria"] = parsed["criteria[]"].map(json.loads)
    parsed["criteria_score"] = parsed["criteria_score[]"].map(json.loads)
    parsed["criteria_max_score"] = parsed["criteria_max_score[]"].map(json.loads)
    parsed["score"] = parsed["student_score"].astype(float)
    parsed["max_score"] = parsed["criteria_max_score"].map(sum)

    problems = []
    for index, row in parsed.iterrows():
        lengths = {len(row["criteria"]), len(row["criteria_score"]), len(row["criteria_max_score"])}
        if len(lengths) != 1:
            problems.append(f"Row {index}: criteria/score/max-score lengths differ")
        elif sum(row["criteria_score"]) != row["score"]:
            problems.append(f"Row {index}: student_score {row['score']} != sum of criterion scores")
    if problems:
        raise ValueError(f"{len(problems)} invalid rows, e.g. {problems[:5]}")
    return raw, parsed


raw_df, df = load_and_validate_dataset(INPUT_PATH)
df["outcome"] = pd.cut(
    df["score"] / df["max_score"], [-1, 0, 0.999, 1], labels=["none met", "partial", "all met"]
).astype(str)

print(f"{len(df)} rows, {df['question'].nunique()} unique questions")
print(df["outcome"].value_counts().to_string())
print(df["criteria"].map(len).value_counts().sort_index().rename_axis("criteria per row").to_string())

3778 rows, 95 unique questions
outcome
partial     1470
all met     1400
none met     908
criteria per row
1     971
2    1212
3     813
4     450
5     332


## Prompts

The earlier runs showed four failure modes that would get baked into the student model:

| Failure | Fix |
|---|---|
| **Generous marks led to invented praise.** Some points are marked MET when the student only loosely said them, or even said the opposite. Told the marks are final, the teacher credited students with claims they never made. | The teacher credits only what the student actually wrote. When a point was loosely expressed, it affirms the idea and then states it precisely. It corrects any wrong claim the student made, even next to a MET point. A valid point the reference doesn't list is treated as valid, not as an error, and the judge counts calling it wrong as a factual error. |
| **One template everywhere.** In the first 27B run, 76% of outputs started with "You correctly identified…" and 77% ended with "Next time, be sure to…". | Each row is assigned an opening and a closing style; a retry draws a new pair. The three stock phrases are rejected by a rule check. The style instruction appears only in the teacher prompt, so the student model learns the whole mix and needs no style input at inference. |
| **Leaked grading mechanics,** such as "The original requirements emphasize…". | The prompt bans grading words and a regex enforces the ban. The prompt contains no example feedback that could be copied verbatim. |
| **Contradictory instructions.** The old prompt told the teacher to say an answer was "complete", and the judge rejected that word as grading language. | The wording is removed from the prompt, and checking grading language is now the regex's job only. |

The **student** prompt used in the SFT export stays deliberately short. The detailed behaviour should be learned from the targets, not read from a long system prompt at inference time.

In [4]:
TEACHER_SYSTEM_PROMPT = """You are an experienced computer science tutor writing feedback on one student's short answer to an exam question.

You are given the question, a reference answer, the student's answer, and the key points the answer was expected to contain. Each key point has been marked MET or NOT MET by the marker. These marks are final: your job is to help the student understand them, not to re-grade.

Write feedback that:
- Speaks directly to the student ("you") in a warm, plain, professional tone.
- Acknowledges what the student actually got right, described the way the student put it. Never credit them with a claim, term or detail they did not write. If a MET point is only loosely expressed, affirm their idea and then state it more precisely instead of pretending they said it exactly.
- Covers every NOT MET point: say what is missing or wrong and explain the correct idea in one or two sentences so the student actually learns it.
- Corrects any specific incorrect claim the student wrote, even one that sits alongside a MET point.
- Treats a technically valid point that the reference answer does not list as valid, not as an error: acknowledge it, then explain what the question was looking for.
- Never says or implies that a MET point is missing.
- When every point is MET, says briefly why the answer works and does not invent extra requirements.
- Uses the opening and closing given at the end of the request.

Rules:
- Base every statement on the question, the reference answer and the student's answer. Never contradict the reference answer.
- Never mention key points, checklists, criteria, MET/NOT MET, marks, scores, grading, or the reference/model/desired answer. Talk about the subject matter, as a tutor would.
- Do not comment on spelling, grammar or length.
- Length: about 30-60 words when every point is MET; otherwise about 50-130 words, scaling with how much is missing.
- Plain prose in one or two short paragraphs. No headings, bullet points, bold text or other markdown.
- Do not open with "You correctly ...". Avoid stock phrasing such as "However, your answer missed" and "Next time, be sure to". Use "However" at most once.

Output only the feedback."""

# One opening and one closing is drawn per row (and redrawn on each retry) so the targets are not one template.
OPENINGS = {
    "their_idea": "Start with the specific idea the student got right, in their own terms (or, if nothing is right, the idea they were reaching for).",
    "concept_first": "Start with one sentence on the core concept the question is about, then connect it to what the student wrote.",
    "overall_take": "Start with a short, plain overall take on how close the answer is, then get specific.",
    "main_gap_first": "Start with the most important thing to fix, then acknowledge what the student did well.",
}
CLOSINGS = {
    "next_step": "End with one concrete instruction for what to add or change in a similar answer.",
    "self_check": "End with one short question the student can ask themselves when answering a similar question.",
    "one_line_answer": "End with a one-sentence version of a complete answer the student could remember.",
    "why_it_works": "End with one sentence on why this answer works.",
}


def choose_style(row_index: int, all_met: bool, attempt: int) -> dict[str, str]:
    rng = random.Random(f"{SEED}-{row_index}-{attempt}")
    openings = ["their_idea", "concept_first", "overall_take"] + ([] if all_met else ["main_gap_first"])
    closings = ["why_it_works", "self_check"] if all_met else ["next_step", "self_check", "one_line_answer"]
    return {"opening": rng.choice(openings), "closing": rng.choice(closings)}


JUDGE_SYSTEM_PROMPT = """You audit tutor feedback written for a student's short answer. You are given the question, a reference answer, the student's answer, the marker's MET/NOT MET decision for each key point, and the feedback. Judge the substance of the feedback, not its wording or style.

Checks:
- covers_missing_points: every NOT MET point is addressed with an explanation of the correct idea (true if no point is NOT MET).
- respects_met_points: it never says or implies that a MET point is missing. Correcting a specific incorrect claim the student actually wrote is allowed.
- faithful_to_student: it never credits the student with a specific claim, term or detail that is absent from, or contradicted by, the student's answer. Paraphrasing is fine, and so is affirming the student's general idea and then stating it more precisely.
- factually_correct: its technical content is correct and consistent with the reference answer. Calling a technically valid student point wrong only because the reference answer does not list it is a factual error.

Reply with only this JSON object:
{"covers_missing_points": true or false, "respects_met_points": true or false, "faithful_to_student": true or false, "factually_correct": true or false, "issues": "the main problem in under 20 words, or an empty string"}"""

STUDENT_SYSTEM_PROMPT = (
    "You are a computer science tutor. Write concise, specific, encouraging feedback to the student on their short answer. "
    + (
        "The marker's decision for each key point is given; help the student understand it. "
        if SFT_INCLUDE_CRITERION_RESULTS
        else "Work out which key points the answer covers. "
    )
    + "Acknowledge what is correct, explain anything missing or incorrect, help the student improve, "
    "and never mention marks, key points or the reference answer."
)

# Records from a different model/prompt are ignored on resume, so editing a prompt never mixes outputs.
PROMPT_VERSION = hashlib.sha1(
    json.dumps(
        [TEACHER_MODEL, TEACHER_THINKING, JUDGE_THINKING, TEACHER_SYSTEM_PROMPT, JUDGE_SYSTEM_PROMPT, OPENINGS, CLOSINGS]
    ).encode()
).hexdigest()[:10]


def criterion_marks(row: pd.Series) -> list[bool]:
    return [score >= maximum for score, maximum in zip(row["criteria_score"], row["criteria_max_score"])]


def build_row_prompt(row: pd.Series, include_marks: bool = True) -> str:
    key_points = []
    for number, (criterion, met) in enumerate(zip(row["criteria"], criterion_marks(row)), start=1):
        mark = ("[MET] " if met else "[NOT MET] ") if include_marks else ""
        key_points.append(f"{number}. {mark}{criterion}")

    sections = [
        f"Question:\n{row['question'].strip()}",
        f"Reference answer:\n{row['desired_answer'].strip()}",
        f"Student answer:\n{row['student_answer'].strip() or '(no answer given)'}",
        "Key points:\n" + "\n".join(key_points),
    ]
    if include_marks:
        sections.append(f"Key points met: {sum(criterion_marks(row))} of {len(row['criteria'])}")
    return "\n\n".join(sections)


def build_teacher_messages(row: pd.Series, style: dict[str, str]) -> list[dict[str, str]]:
    request = (
        f"{build_row_prompt(row)}\n\n"
        f"Opening: {OPENINGS[style['opening']]}\n"
        f"Closing: {CLOSINGS[style['closing']]}"
    )
    return [
        {"role": "system", "content": TEACHER_SYSTEM_PROMPT},
        {"role": "user", "content": request},
    ]


def build_judge_messages(row: pd.Series, feedback: str) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
        {"role": "user", "content": f"{build_row_prompt(row)}\n\nFeedback to audit:\n{feedback}"},
    ]


example_row = df.iloc[0]
print(f"Prompt version: {PROMPT_VERSION}\n")
print(build_teacher_messages(example_row, choose_style(0, all(criterion_marks(example_row)), 1))[1]["content"])

Prompt version: f4791f8a71

Question:
What is the role of a prototype program in problem solving?

Reference answer:
To simulate the behaviour of portions of the desired software product.

Student answer:
High risk problems are address in the prototype program to make sure that the program is feasible.  A prototype may also be used to show a company that the software can be possibly programmed.

Key points:
1. [NOT MET] Answer mentions simulating behaviour of portions of desired software

Key points met: 0 of 1

Opening: Start with a short, plain overall take on how close the answer is, then get specific.
Closing: End with one concrete instruction for what to add or change in a similar answer.


## Load the teacher model

FP8 weights take about 29 GB. At 90% utilisation, that leaves about 13 GB on the L40S for KV cache and batching. Qwen3.5 also needs one Mamba state block per concurrent sequence, and vLLM's default of 256 sequences doesn't fit, so `MAX_NUM_SEQS` caps the batch. If you hit OOM or a "Mamba cache blocks" error, lower `MAX_NUM_SEQS`.

In [5]:
# FlashInfer's top-k/top-p sampler is JIT-compiled with nvcc on first use and needs CUDA >= 12.8;
# the cluster's CUDA 12.4 module rejects its flags. vLLM's PyTorch sampler gives the same results.
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import torch
from vllm import LLM, SamplingParams

properties = torch.cuda.get_device_properties(0)
print(f"GPU: {properties.name}, {properties.total_memory / 1e9:.0f} GB")

llm = LLM(
    model=TEACHER_MODEL,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_num_seqs=MAX_NUM_SEQS,
    limit_mm_per_prompt={"image": 0, "video": 0},  # text-only use of a vision-language checkpoint
    seed=SEED,
)


def generation_params(attempt: int) -> SamplingParams:
    # Qwen's recommended sampling settings; a new seed per attempt so retries differ.
    if TEACHER_THINKING:
        settings = {"temperature": 1.0, "top_p": 0.95}
    else:
        settings = {"temperature": 0.7, "top_p": 0.8}
    return SamplingParams(
        **settings,
        top_k=20,
        min_p=0.0,
        presence_penalty=1.5,
        max_tokens=MAX_FEEDBACK_TOKENS,
        seed=SEED + attempt,
    )


if JUDGE_THINKING:
    JUDGE_PARAMS = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, max_tokens=MAX_JUDGE_TOKENS, seed=SEED)
else:
    JUDGE_PARAMS = SamplingParams(temperature=0.0, max_tokens=MAX_JUDGE_TOKENS)


def run_chat(conversations, params, thinking: bool) -> list[tuple[str, str]]:
    """Batch chat completion; returns (text, finish_reason) with any reasoning block removed."""
    outputs = llm.chat(conversations, params, chat_template_kwargs={"enable_thinking": thinking}, use_tqdm=True)
    results = []
    for output in outputs:
        completion = output.outputs[0]
        text = completion.text.split("</think>")[-1]
        text = re.sub(r"\n\s*\n+", "\n\n", text).strip()
        results.append((text, completion.finish_reason))
    return results

/common/home/projectgrps/CS480/CS480G3/jupyterlab-venv-pytorch-240/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU: NVIDIA L40S, 48 GB
(MainProcess pid=1275643) INFO 10-31 02:22:10 [api_utils.py:286] non-default args: {'seed': 42, 'max_model_len': 16384, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 128, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 0, 'video': 0}, 'model': 'Qwen/Qwen3.5-27B-FP8'}


(MainProcess pid=1275643) INFO 10-31 02:22:14 [model.py:699] Resolved architecture: Qwen3_5ForConditionalGeneration


(MainProcess pid=1275643) INFO 10-31 02:22:14 [model.py:2109] Using max model len 16384


(MainProcess pid=1275643) WARNING 10-31 02:22:14 [model.py:1069] Model does not support mm_device_do_normalize, forcing mm_device_do_normalize = False.


(MainProcess pid=1275643) INFO 10-31 02:22:18 [model.py:917] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.


... (88 outputs left out)

(EngineCore pid=1275719) INFO 10-31 02:24:48 [model_runner.py:1087] Graph capturing finished in 17 secs, took 0.24 GiB
(EngineCore pid=1275719) INFO 10-31 02:24:48 [gpu_worker.py:892] CUDA graph pool memory: 0.24 GiB (actual), 0.55 GiB (estimated), difference: 0.31 GiB (131.1%).
(EngineCore pid=1275719) INFO 10-31 02:24:48 [gpu_worker.py:955] Free memory on device (43.97/44.39 GiB) on startup. Desired GPU memory utilization is (0.9, 39.95 GiB). Actual usage is 28.3 GiB for consumed memory (weights + non-torch), 3.16 GiB for peak activation, and 0.24 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=8707007693` (8.11 GiB) to fit into requested memory, or `--kv-cache-memory=13018437632` (12.12 GiB) to fully utilize gpu memory. Current kv cache memory in use is 8.49 GiB.


(EngineCore pid=1275719) INFO 10-31 02:24:49 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.


(EngineCore pid=1275719) INFO 10-31 02:24:49 [core.py:373] init engine (profile, create kv cache, warmup model) took 121.15 s (compilation: 23.18 s)
(EngineCore pid=1275719) INFO 10-31 02:24:49 [kv_cache_utils.py:772] kv cache group sizes [784, 784, 784, 784]
(EngineCore pid=1275719) INFO 10-31 02:24:49 [kv_cache_utils.py:773] kv lcm block sizes 784
(EngineCore pid=1275719) INFO 10-31 02:24:49 [kernel.py:408] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])
(EngineCore pid=1275719) INFO 10-31 02:24:49 [compilation.py:331] Enabled custom fusions: norm_quant, act_quant


(MainProcess pid=1275643) INFO 10-31 02:24:58 [hf.py:646] Detected the chat template content format to be 'openai'. You can set `--chat-template-content-format` to override this.


## Quality checks

The **rule checks** are cheap and deterministic. They catch:
- Output that is truncated, too short or too long.
- Markdown formatting, or non-English text.
- Grading language, such as "criteria", "full marks" or "reference answer".
- The stock phrases from the first run (an opening "You correctly…", "However, your answer missed", "Next time, be sure to"), and "However" used more than once.

The **judge** is the same model, called in thinking mode with a separate auditing prompt. It checks only substance:
- Every NOT MET point is covered.
- No MET point is treated as missing.
- The student is not credited with claims they didn't write.
- The technical content is correct.

In the first run, the judge also checked wording itself. It rejected 826 good rows over everyday words like "complete", and almost half of the all-met rows were lost. That check now belongs to the regex alone.

In [6]:
# Narrow patterns: question stems contain phrases like "no marks", and CS text can say "the OS marks a page".
GRADING_TERMS = re.compile(
    r"\b(criteri(on|a)|rubric|checklist|key points|not met|scor(e|es|ed|ing)|grad(e|ed|ing)|marker|"
    r"(full|no|lose|lost|earn|earned|award|awarded) marks?|(reference|model|desired|expected|sample) answer)\b",
    re.IGNORECASE,
)
# The phrases the first run collapsed onto; the student model would copy them.
# "You correctly ..." reads naturally mid-feedback, so only an opening one is rejected.
STOCK_PHRASES = re.compile(
    r"^\W*you correctly\b|\bhowever, your (answer|explanation) (missed|did not|didn't)\b|\bnext time, be sure to\b",
    re.IGNORECASE,
)
MARKDOWN = re.compile(r"\*\*|__|^\s*#|^\s*[-*•]\s|^\s*\d+[.)]\s", re.MULTILINE)
NON_ENGLISH = re.compile(r"[　-鿿가-힯]")
JUDGE_KEYS = [
    "covers_missing_points",
    "respects_met_points",
    "faithful_to_student",
    "factually_correct",
]


def rule_issues(text: str, finish_reason: str) -> list[str]:
    issues = []
    words = len(text.split())
    if finish_reason == "length":
        issues.append("truncated")
    if words < 15:
        issues.append(f"too short ({words} words)")
    if words > 200:
        issues.append(f"too long ({words} words)")
    if match := GRADING_TERMS.search(text):
        issues.append(f"grading language: {match.group(0)!r}")
    if match := STOCK_PHRASES.search(text):
        issues.append(f"stock phrase: {match.group(0)!r}")
    if len(re.findall(r"\bHowever\b", text)) > 1:
        issues.append("repeated 'However'")
    if MARKDOWN.search(text):
        issues.append("markdown formatting")
    if NON_ENGLISH.search(text):
        issues.append("non-English text")
    return issues


def parse_judgement(text: str) -> dict | None:
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        verdict = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    if not all(isinstance(verdict.get(key), bool) for key in JUDGE_KEYS):
        return None
    return verdict

## Generation pipeline

Every generation and judgement is appended to JSONL as soon as its chunk finishes. Re-running `run_pipeline` picks up where it stopped. Each row moves through these states:
- `pending`: nothing generated yet.
- `unjudged`: generated and passed the rule checks; waiting for the judge.
- `failed`: failed a rule check or the judge. Regenerated with a new seed until `MAX_ATTEMPTS` is reached.
- `passed`: kept for the SFT export.

In [7]:
def chunks(items: list, size: int):
    for start in range(0, len(items), size):
        yield items[start : start + size]


def append_jsonl(path: Path, records: list[dict]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_latest(path: Path) -> dict[int, dict]:
    """Latest record per row for the current PROMPT_VERSION."""
    latest = {}
    if not path.exists():
        return latest
    with path.open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            record = json.loads(line)
            if record["prompt_version"] != PROMPT_VERSION:
                continue
            current = latest.get(record["row_index"])
            if current is None or record["attempt"] >= current["attempt"]:
                latest[record["row_index"]] = record
    return latest


def row_status(row_index: int, generations: dict, judgements: dict) -> str:
    generation = generations.get(row_index)
    if generation is None:
        return "pending"
    if generation["rule_issues"]:
        return "failed"
    judgement = judgements.get(row_index)
    if judgement is None or judgement["attempt"] != generation["attempt"]:
        return "unjudged"
    return "passed" if judgement["passed"] else "failed"


def generate_feedback(row_indices: list[int], attempts: dict[int, int]) -> None:
    for batch in chunks(row_indices, CHUNK_SIZE):
        started = time.perf_counter()
        rows = {index: df.loc[index] for index in batch}
        styles = {index: choose_style(index, all(criterion_marks(rows[index])), attempts[index]) for index in batch}
        conversations = [build_teacher_messages(rows[index], styles[index]) for index in batch]
        params = [generation_params(attempts[index]) for index in batch]
        results = run_chat(conversations, params, TEACHER_THINKING)
        append_jsonl(
            GENERATIONS_PATH,
            [
                {
                    "row_index": index,
                    "attempt": attempts[index],
                    "prompt_version": PROMPT_VERSION,
                    "style": styles[index],
                    "feedback": text,
                    "finish_reason": finish_reason,
                    "rule_issues": rule_issues(text, finish_reason),
                }
                for index, (text, finish_reason) in zip(batch, results)
            ],
        )
        print(f"Generated {len(batch)} rows in {time.perf_counter() - started:.0f}s")


def judge_feedback(row_indices: list[int], generations: dict[int, dict]) -> None:
    for batch in chunks(row_indices, CHUNK_SIZE):
        started = time.perf_counter()
        conversations = [build_judge_messages(df.loc[index], generations[index]["feedback"]) for index in batch]
        results = run_chat(conversations, JUDGE_PARAMS, JUDGE_THINKING)
        records = []
        for index, (text, _) in zip(batch, results):
            verdict = parse_judgement(text)
            records.append(
                {
                    "row_index": index,
                    "attempt": generations[index]["attempt"],
                    "prompt_version": PROMPT_VERSION,
                    "passed": verdict is not None and all(verdict[key] for key in JUDGE_KEYS),
                    "verdict": verdict,
                    "raw_judgement": None if verdict is not None else text,
                }
            )
        append_jsonl(JUDGEMENTS_PATH, records)
        print(f"Judged {len(batch)} rows in {time.perf_counter() - started:.0f}s")


def run_pipeline(row_indices=None, max_attempts: int = MAX_ATTEMPTS) -> pd.Series:
    targets = [int(index) for index in (df.index if row_indices is None else row_indices)]
    while True:
        generations = load_latest(GENERATIONS_PATH)
        judgements = load_latest(JUDGEMENTS_PATH)
        statuses = {index: row_status(index, generations, judgements) for index in targets}

        to_judge = [index for index in targets if statuses[index] == "unjudged"]
        if to_judge:
            print(f"Judging {len(to_judge)} rows")
            judge_feedback(to_judge, generations)
            continue

        to_generate = [
            index
            for index in targets
            if statuses[index] == "pending"
            or (statuses[index] == "failed" and generations[index]["attempt"] < max_attempts)
        ]
        if not to_generate:
            return pd.Series(statuses).value_counts()

        attempts = {index: generations[index]["attempt"] + 1 if index in generations else 1 for index in to_generate}
        print(f"Generating {len(to_generate)} rows (attempt {min(attempts.values())}-{max(attempts.values())})")
        generate_feedback(to_generate, attempts)


def show_rows(row_indices) -> None:
    generations = load_latest(GENERATIONS_PATH)
    judgements = load_latest(JUDGEMENTS_PATH)
    for index in (int(index) for index in row_indices):
        generation = generations.get(index, {})
        judgement = judgements.get(index, {})
        style = generation.get("style", {})
        print("=" * 100)
        print(build_row_prompt(df.loc[index]))
        print(
            f"\n--- Feedback (attempt {generation.get('attempt')}, {row_status(index, generations, judgements)}, "
            f"{style.get('opening')} / {style.get('closing')}) ---"
        )
        print(generation.get("feedback", ""))
        if generation.get("rule_issues"):
            print(f"Rule issues: {generation['rule_issues']}")
        if judgement.get("attempt") == generation.get("attempt") and not judgement["passed"]:
            print(f"Judge: {judgement['verdict'] or judgement['raw_judgement']}")

## Smoke test

Generate and judge 12 rows: 4 where no criteria were met, 4 partial, 4 where all were met. Read them before launching the full run. If you change a prompt, the prompt version changes and these rows are redone automatically.

In [8]:
smoke_rows = df.groupby("outcome").sample(n=4, random_state=SEED).index
print(run_pipeline(smoke_rows).to_string())
show_rows(smoke_rows)

passed    11
failed     1
Question:
How can you implement a queue with an array?

Reference answer:
Use a circular array. Keep the rear of the queue toward the end of the array, and the front toward the beginning, and allow the rear pointer to wrap around.

Student answer:
your storage class has an array, and you keep track of the index of the first and last item in the array, and wrap around when end of the array is full

Key points:
1. [MET] Answer mentions circular array
2. [MET] Answer mentions rear toward end, front toward beginning

Key points met: 2 of 2

--- Feedback (attempt 1, passed, overall_take / why_it_works) ---
Your answer captures the core strategy well by describing a storage class with an array and tracking indices for the first and last items. You correctly identified that wrapping around when the end is reached is essential to maintain this structure. This approach works because it allows the queue to reuse empty space efficiently without shifting elements.
Questio

## Full run

All 3,778 rows. Thinking mode for the judge makes judging the slowest step, roughly several times the cost of generating. The run is safe to interrupt: re-running this cell resumes from the last checkpointed chunk.

Records from the first run have a different prompt version, so they are ignored rather than mixed in. Keep a copy of the old exports first, because this run overwrites the CSV and SFT files.

In [9]:
started = time.perf_counter()
summary = run_pipeline()
print(summary.to_string())
print(f"Elapsed: {(time.perf_counter() - started) / 60:.1f} min")

Judging 2015 rows


Rendering conversations:   0%|          | 0/512 [00:00<?, ?it/s]

Rendering conversations:  20%|██        | 103/512 [00:00<00:00, 1026.22it/s]

Rendering conversations:  47%|████▋     | 239/512 [00:00<00:00, 1218.15it/s]

Rendering conversations:  72%|███████▏  | 371/512 [00:00<00:00, 1264.04it/s]

Rendering conversations:  99%|█████████▉| 509/512 [00:00<00:00, 1309.21it/s]

... (5,497 outputs left out)

Processed prompts: 100%|██████████| 211/211 [45:19<00:00, 36.90s/it, est. speed input: 43.29 toks/s, output: 318.45 toks/s]

Processed prompts: 100%|██████████| 211/211 [45:19<00:00, 36.90s/it, est. speed input: 43.29 toks/s, output: 318.45 toks/s]

Processed prompts: 100%|██████████| 211/211 [45:19<00:00, 12.89s/it, est. speed input: 43.29 toks/s, output: 318.45 toks/s]

Judged 211 rows in 2720s
passed    3533
failed     245
Elapsed: 576.8 min


## Review

Check these before exporting:
- **Kept vs source outcome mix:** the shares should be within a few points. If too few all-met rows survive, the student model learns to find faults where there are none.
- **Top opener share:** no single three-word opener should cover more than about 10% of rows. The student model copies whatever template dominates.
- **Failure reasons:** what the remaining rejections were for.
- **Style mix:** how the assigned openings and closings are spread.

In [10]:
def collect_results() -> pd.DataFrame:
    generations = load_latest(GENERATIONS_PATH)
    judgements = load_latest(JUDGEMENTS_PATH)
    rows = []
    for index in df.index:
        generation = generations.get(index)
        judgement = judgements.get(index)
        if judgement is not None and generation is not None and judgement["attempt"] == generation["attempt"]:
            verdict = judgement["verdict"]
            judge_issues = "unparsable judgement" if verdict is None else ", ".join(k for k in JUDGE_KEYS if not verdict[k])
        else:
            judge_issues = ""
        style = generation.get("style", {}) if generation else {}
        rows.append(
            {
                "row_index": index,
                "feedback": generation["feedback"] if generation else "",
                "attempts": generation["attempt"] if generation else 0,
                "status": row_status(index, generations, judgements),
                "opening": style.get("opening", ""),
                "closing": style.get("closing", ""),
                "rule_issues": "; ".join(generation["rule_issues"]) if generation else "",
                "judge_issues": judge_issues,
            }
        )
    return pd.DataFrame(rows).set_index("row_index")


results = collect_results()
results["words"] = results["feedback"].str.split().str.len()
kept = results[results["status"] == "passed"]

print(results["status"].value_counts().to_string(), "\n")
print(results["attempts"].value_counts().sort_index().rename_axis("attempts used").to_string(), "\n")
print(
    pd.DataFrame(
        {
            "source": df["outcome"].value_counts(normalize=True),
            "kept": df.loc[kept.index, "outcome"].value_counts(normalize=True),
            "kept rate": df.loc[kept.index, "outcome"].value_counts() / df["outcome"].value_counts(),
        }
    ).round(3),
    "\n",
)
print(kept.join(df["outcome"]).groupby("outcome")["words"].describe().round(0), "\n")

failed = results[results["status"] != "passed"]
if len(failed):
    reasons = pd.concat(
        [failed["rule_issues"].str.split("; ").explode(), failed["judge_issues"].str.split(", ").explode()]
    )
    print(reasons[reasons != ""].str.replace(r":.*", "", regex=True).value_counts().head(10).to_string(), "\n")

openers = kept["feedback"].str.split().str[:3].str.join(" ")
however_share = kept["feedback"].str.contains(r"\bHowever\b").mean()
print(f"Top opener covers {openers.value_counts(normalize=True).iloc[0]:.1%} of kept rows")
print(openers.value_counts().head(10).to_string(), "\n")
print(f"'However' appears in {however_share:.1%} of kept rows\n")
print(pd.crosstab(kept["opening"], kept["closing"], margins=True))

status
passed    3533
failed     245 

attempts used
1    2319
2     769
3     309
4     381 

          source   kept  kept rate
outcome                           
partial    0.389  0.378      0.910
all met    0.371  0.369      0.930
none met   0.240  0.253      0.985 

           count  mean   std   min   25%   50%   75%    max
outcome                                                    
all met   1302.0  62.0  10.0  37.0  55.0  61.0  68.0  109.0
none met   894.0  80.0  15.0  46.0  69.0  77.0  87.0  141.0
partial   1337.0  82.0  14.0  42.0  72.0  80.0  90.0  140.0 

stock phrase             169
faithful_to_student       43
respects_met_points       18
unparsable judgement       9
factually_correct          7
grading language           2
covers_missing_points      2 

Top opener covers 18.6% of kept rows
feedback
Your answer correctly     658
This question tests       349
Your answer is            229
Your answer captures      145
This question asks        139
The question asks        

In [11]:
# Spot-check passed rows across outcomes, then any failures.
show_rows(results[results["status"] == "passed"].join(df["outcome"]).groupby("outcome").sample(n=2, random_state=SEED).index)
show_rows(failed.index[:3])

Question:
What is the main advantage of arrays over linked lists?

Reference answer:
The elements in an array can be accessed directly (as opposed to linked lists, which require iterative traversal).

Student answer:
Any entitiy in an array may accessed immediately, whereas a linked list must be traversed one item at a time because each object only points adjacent objects

Key points:
1. [MET] Answer mentions direct/random access to elements

Key points met: 1 of 1

--- Feedback (attempt 1, passed, overall_take / why_it_works) ---
Your response captures the core distinction between arrays and linked lists very well. You correctly identified that array elements can be accessed immediately, while linked lists require traversing items one by one due to their pointer structure. This answer works because it clearly contrasts direct access with the sequential traversal needed for linked lists.
Question:
What is a binary search tree?

Reference answer:
A binary tree that has the property that

## Export

- **CSV**: the original 8 columns plus `feedback` and `feedback_passed_checks`, following the format of the earlier notebooks.
- **SFT JSONL**: only rows that passed every check, in chat `messages` format with the short student system prompt. The split is by **question**, so validation measures generalisation to unseen questions instead of memorised reference answers.

In [12]:
export = raw_df.copy()
export["feedback"] = results["feedback"]
export["feedback_passed_checks"] = results["status"].eq("passed")
export.to_csv(OUTPUT_CSV_PATH, index=False, quoting=csv.QUOTE_MINIMAL)
print(f"CSV: {OUTPUT_CSV_PATH} ({export['feedback_passed_checks'].sum()}/{len(export)} passed)")

questions = sorted(df["question"].unique())
random.Random(SEED).shuffle(questions)
val_questions = set(questions[: max(1, round(len(questions) * VAL_FRACTION))])


def to_sft_record(index: int) -> dict:
    return {
        "messages": [
            {"role": "system", "content": STUDENT_SYSTEM_PROMPT},
            {"role": "user", "content": build_row_prompt(df.loc[index], include_marks=SFT_INCLUDE_CRITERION_RESULTS)},
            {"role": "assistant", "content": results.at[index, "feedback"]},
        ]
    }


passed = results.index[results["status"] == "passed"]
splits = {
    SFT_TRAIN_PATH: [index for index in passed if df.at[index, "question"] not in val_questions],
    SFT_VAL_PATH: [index for index in passed if df.at[index, "question"] in val_questions],
}
for path, indices in splits.items():
    with path.open("w", encoding="utf-8") as handle:
        for index in indices:
            handle.write(json.dumps(to_sft_record(index), ensure_ascii=False) + "\n")
    print(f"{path.name}: {len(indices)} examples")
print(f"Validation questions: {len(val_questions)} of {len(questions)}")

CSV: /common/home/projectgrps/CS480/CS480G3/Datasets/teacher_feedback/training_data_moh_eng_rubrics_with_feedback.csv (3533/3778 passed)


sft_train.jsonl: 3238 examples
sft_val.jsonl: 295 examples
Validation questions: 10 of 95


## Next: QLoRA on the student

Train the student with `qlora_feedback_qwen3.5-9B.ipynb`. It reads `training_data_moh_eng_rubrics_with_feedback.csv` and re-splits by question with the grading notebook's 80/10/10 split, so both adapters share one unseen test set.

`sft_train.jsonl` and `sft_val.jsonl` use this notebook's own 90/10 split, which includes the grading model's test questions. Use them only when a different split doesn't matter.

After training, come back to the section below to judge the student's test outputs.

## Judge the student model

Scores the test-set feedback written by the QLoRA student and its base model (`student_test_predictions.jsonl`, written by `qlora_feedback_qwen3.5-9B.ipynb`). It uses the same judge and the same four checks the teacher's feedback had to pass.

To run it, run the setup cells, the model cell, the rule-check cell and the pipeline-functions cell, then this one. Skip the smoke test and the full run. Judgements are checkpointed per chunk, so an interrupted run resumes.

The teacher's kept feedback passed this judge by construction, so there is no fair teacher score on the same rows. Compare base against adapter.

In [ ]:
STUDENT_PREDICTIONS_PATH = OUTPUT_DIR / "student_test_predictions.jsonl"
STUDENT_JUDGEMENTS_PATH = OUTPUT_DIR / "student_test_judgements.jsonl"


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


predictions = read_jsonl(STUDENT_PREDICTIONS_PATH)
done = {(record["model"], record["row_index"], record["feedback"]) for record in read_jsonl(STUDENT_JUDGEMENTS_PATH)}
pending = [p for p in predictions if (p["model"], p["row_index"], p["feedback"]) not in done]
print(f"{len(predictions)} student outputs, {len(pending)} still to judge")

for batch in chunks(pending, CHUNK_SIZE):
    started = time.perf_counter()
    outputs = run_chat(
        [build_judge_messages(df.loc[p["row_index"]], p["feedback"]) for p in batch], JUDGE_PARAMS, JUDGE_THINKING
    )
    records = []
    for prediction, (text, _) in zip(batch, outputs):
        verdict = parse_judgement(text)
        records.append(
            {
                "model": prediction["model"],
                "row_index": prediction["row_index"],
                "feedback": prediction["feedback"],
                "parsed": verdict is not None,
                "passed": verdict is not None and all(verdict[key] for key in JUDGE_KEYS),
                **{key: verdict is not None and verdict[key] for key in JUDGE_KEYS},
                "issues": verdict.get("issues", "") if verdict else "unparsable judgement",
            }
        )
    append_jsonl(STUDENT_JUDGEMENTS_PATH, records)
    print(f"Judged {len(batch)} outputs in {time.perf_counter() - started:.0f}s")

current = {(p["model"], p["row_index"], p["feedback"]) for p in predictions}
student_judged = pd.DataFrame(
    [r for r in read_jsonl(STUDENT_JUDGEMENTS_PATH) if (r["model"], r["row_index"], r["feedback"]) in current]
).drop_duplicates(subset=["model", "row_index"], keep="last")
student_judged["outcome"] = student_judged["row_index"].map(df["outcome"])

print("\nPass rate per check")
print(student_judged.groupby("model")[JUDGE_KEYS + ["passed", "parsed"]].mean().round(3).T.to_string())
print("\nOverall pass rate by outcome")
print(student_judged.pivot_table(index="outcome", columns="model", values="passed", aggfunc="mean").round(3).to_string())
print("\nMost common issues for the adapter")
print(student_judged.query("model == 'qlora' and not passed")["issues"].value_counts().head(10).to_string())